In [ ]:
from fmsae_repro import MODELS, RESULTS, LEGACY_ANALYSIS, load_manifest

import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['pdf.fonttype'] = 42

### MP-SAE feature selection and clustering

This notebook repeats the SAE feature analysis for the matching pursuit SAE (MP-SAE). The MP-SAE has an 8x expansion of the 1,024-dimensional UNI embedding, which gives 8,192 features. Its encoder is greedy matching pursuit over a learned dictionary, applied to the raw (unscaled) UNI patch embeddings.

We do not train a new model. We load the checkpoint used in the manuscript, which `scripts/legacy_conversion/convert_mpsae.py` converted to `mesoslide.tools.MatchingPursuitSAE`. Feature filtering and clustering use the same thresholds as the original analysis.

In [ ]:
import ezslide
import mesoslide as ms
import numpy as np
import joblib

OUT = RESULTS / "figures" / "MPSAE"
OUT.mkdir(parents=True, exist_ok=True)

First, we load the slides, which contain the UNI patch embeddings, and the converted MP-SAE model.

In [ ]:
manifest = load_manifest()
slides = ezslide.open_slides(manifest, attach_images=False)

mpsae = joblib.load(MODELS / "MPSAE" / "mpsae_model.joblib")
mpsae

Next, we apply the MP-SAE to every slide and store the feature scores in each tile table under the prefix `UNI_MPSAE`. The existing `UNI_SAE` columns are kept. Slides that already have `UNI_MPSAE` columns are skipped, so re-running this cell is cheap.

This step was first checked on a copy of one store with `scripts/features/dryrun_mpsae_write.py`. Matching pursuit is iterative, so run it on a GPU.

In [ ]:
for _, slide in slides.items():
    ms.tl.feature_extraction(
        slide,
        model="uni",
        tile_key="tiles",
        key_added="UNI_embedding",
        sparse=True,
        sparse_transform=lambda X: mpsae.transform(X, device="cuda"),
        sparse_key_added="UNI_MPSAE",
        save=True,
    )

We select features that are active in more than 1% of patches and whose maximum score exceeds 8.6. These are the thresholds used for the MP-SAE in the original analysis (`params2.toml`).

In [ ]:
from mesoslide.tools import FeatureSelector

selector = FeatureSelector(
    pct_threshold=0.01,
    max_score_threshold=8.6,
    n_chunks=40
)
selector.compute_activation_stats(slides,
                                  feature_prefix='UNI_MPSAE',
                                  num_features=mpsae.embed_dim_)
g = selector.plot_feature_selection()
g.savefig(OUT / "feature_selection.pdf", dpi=300)

In [ ]:
highly_active_features_idx = selector.get_selected_indices()
len(highly_active_features_idx)

We compare the selected features with the 211 features selected in the original analysis. Small differences are expected, because the current tiling gives about 2.7% fewer patches.

In [ ]:
legacy = np.load(LEGACY_ANALYSIS / "SAE_paper" / "TB" / "3D_HnE" / "mp_sae" / "reordered_active_features.npz")
legacy_idx = set(legacy["reordered_active_feature_idx"].tolist())
new_idx = set(highly_active_features_idx.tolist())
print(f"original: {len(legacy_idx)}, current: {len(new_idx)}, shared: {len(legacy_idx & new_idx)}, "
      f"Jaccard: {len(legacy_idx & new_idx) / len(legacy_idx | new_idx):.3f}")

We cluster the selected features by the co-occurrence of their strong activations. Scores are normalized by each feature's maximum and binarized at 0.5. Pairwise IoU is computed over all patches, and average-linkage hierarchical clustering is cut into 25 clusters, as in the original analysis.

In [ ]:
from mesoslide.tools import FeatureClusterer

clusterer = FeatureClusterer(high_activation_threshold=0.5)
clusterer.compute_iou(slides, feature_prefix='UNI_MPSAE',
                      feature_indices=highly_active_features_idx)
clusterer.cluster(threshold=25, criterion='maxclust', linkage_method='average')

In [ ]:
g = clusterer.plot_heatmap()
g.savefig(OUT / "feature_pairwise_iou.pdf", dpi=300)

In [ ]:
np.savez_compressed(
    OUT / "reordered_active_features.npz",
    reordered_active_feature_idx=clusterer.get_reordered_feature_indices(),
    reordered_clusters=clusterer.get_cluster_assignments(),
)

Finally, we select the top-scoring patch for each feature and plot the feature gallery, colored by cluster.

In [ ]:
_ = clusterer.select_exemplars(slides)

In [ ]:
import distinctipy
from matplotlib.colors import ListedColormap

cmap = ListedColormap(distinctipy.get_colors(25, rng=43))

fig, axs = clusterer.plot_feature_gallery(
    cmap=cmap,
    fontsize=10,
    return_fig=True,
)
fig.savefig(OUT / "feature_gallery_clustered_by_iou_with_title.pdf", dpi=300)